# Transmission Line Pulse Propagation

Simulation of pulse propagation through a lossy transmission line using the `TransmissionLine` block. We observe the propagation delay and attenuation of a Gaussian pulse traveling through the line.

## Transmission Line Model

The `TransmissionLine` block models wave propagation in the scattering domain:

$$b_1(t) = T \cdot a_2(t - \tau)$$
$$b_2(t) = T \cdot a_1(t - \tau)$$

where $\tau = L / v_p$ is the propagation delay, $v_p = c_0 / \sqrt{\varepsilon_r}$ is the phase velocity, and $T = 10^{-\alpha L / 20}$ is the voltage transmission coefficient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope
from pathsim.solvers import RKCK54

from pathsim_rf import TransmissionLine

## System Setup

We send a Gaussian pulse into port 1 of a transmission line and observe the delayed output at port 2. The line has a relative permittivity of 4 (typical for FR-4 PCB material).

In [ ]:
# Transmission line parameters
length = 0.3     # 30 cm line
er = 4.0         # FR-4 permittivity
Z0 = 50.0        # Characteristic impedance [Ohm]

# Calculate expected delay
C0 = 299792458.0
vp = C0 / np.sqrt(er)
tau = length / vp
print(f'Phase velocity: {vp:.2e} m/s')
print(f'Propagation delay: {tau*1e9:.2f} ns')

In [ ]:
# Gaussian pulse parameters
t_center = 3e-9   # Pulse center [s]
sigma = 0.5e-9    # Pulse width [s]

def gaussian_pulse(t):
    return np.exp(-0.5 * ((t - t_center) / sigma) ** 2)

# Blocks
src = Source(func=gaussian_pulse)
tl = TransmissionLine(length=length, er=er, attenuation=0.0, Z0=Z0)
sco = Scope(labels=['Input (a1)', 'Output (b2)'])

# Connect: source -> port a1, observe input and output (b2)
connections = [
    Connection(src, tl[0], sco[0]),
    Connection(tl["b2"], sco[1]),
]

sim = Simulation(
    [src, tl, sco],
    connections,
    dt=0.01e-9,
    Solver=RKCK54
)

sim.run(10e-9)

## Lossless Propagation

The output pulse is a delayed copy of the input with the same amplitude. The delay matches the expected propagation time.

In [ ]:
sco.plot()
plt.show()

## Effect of Attenuation

Now let's compare different attenuation values to see how losses affect the transmitted pulse. We run separate simulations for each attenuation level and overlay the results.

In [ ]:
attenuations = [0.0, 5.0, 10.0, 20.0]  # dB/m

fig, ax = plt.subplots(dpi=120)

for atten in attenuations:
    src = Source(func=gaussian_pulse)
    tl = TransmissionLine(length=length, er=er, attenuation=atten, Z0=Z0)
    sco = Scope()

    sim = Simulation(
        [src, tl, sco],
        [Connection(src, tl[0]), Connection(tl["b2"], sco)],
        dt=0.01e-9,
        Solver=RKCK54
    )

    sim.run(10e-9)

    t, [y] = sco.read()
    total_loss = atten * length
    ax.plot(np.array(t) * 1e9, y, 
            label=f'{atten} dB/m ({total_loss:.1f} dB total)', linewidth=2)

ax.set_xlabel('Time [ns]')
ax.set_ylabel('Voltage [V]')
ax.set_title('Effect of Attenuation on Pulse Propagation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()